In [3]:
# Step 1: Import required libraries
import pandas as pd
import string

# Load your datasets
places = pd.read_csv("places.csv")
reviews = pd.read_csv("reviews.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'places.csv'

In [ ]:
places.head()

In [ ]:
reviews.head()

## Data Preprocessing

Group reviews

In [ ]:
# Group reviews to summarize per place_id
reviews_grouped = (
    reviews
    .groupby("place_id", as_index=False)
    .agg({
        "review_id": "count",            # how many reviews per shop
        "text": lambda x: list(x)        # optional: keep list of review texts
    })
    .rename(columns={"review_id": "review_count"})
)
reviews_grouped.head()


In [ ]:
# Merge grouped reviews with places
shops_r = reviews_grouped.merge(
    places[["place_id", "name", "address", "lat", "lng", "user_ratings_total"]],
    on="place_id",
    how="inner",
    validate="1:1"   # now one row per place
)
shops_r.head()


### Quick overview of the data

In [ ]:
print("Places dataset-")
print(f"Rows: {places.shape[0]}, Columns: {places.shape[1]}")

print("Reviews dataset-")
print(f"Rows: {reviews.shape[0]}, Columns: {reviews.shape[1]}")

print("Grouped reviews dataset-")
print(f"Rows: {reviews_grouped.shape[0]}, Columns: {reviews_grouped.shape[1]}")

print("Merged dataset-")
print(f"Rows: {shops_r.shape[0]}, Columns: {shops_r.shape[1]}")

**Check for missingness**

In [ ]:
# Count missing values in each column
shops_r.isna().sum()


**Check for duplicates**

In [ ]:
# Verify one unique row per place
duplicate_places = shops_r["place_id"].duplicated().sum()
print(f"Duplicate place_ids: {duplicate_places}")


In [ ]:
print("Max reviews seen:", shops_r["review_count"].max())
# Ensure we have a time column to sort by
if "publish_time_utc" in reviews.columns:
    reviews["publish_time_utc"] = pd.to_datetime(reviews["publish_time_utc"], errors="coerce", utc=True)
else:
    reviews["publish_time_utc"] = pd.NaT

# Deduplicate by review_id if present
if "review_id" in reviews.columns:
    reviews = reviews.drop_duplicates(subset=["review_id"])

# Keep the latest 5 per place
reviews_capped = (
    reviews
    .sort_values(["place_id", "publish_time_utc"])
    .groupby("place_id", as_index=False, sort=False)
    .tail(5)
    .reset_index(drop=True)
)

# Then build shops_r from reviews_capped (not from the full reviews)
# Example:
shops_r = (
    reviews_capped
    .groupby(["place_id","name","lat","lng","category"], dropna=False, as_index=False)
    .agg(
        review_count=("review_id","nunique"),
        avg_rating=("rating","mean"),
        # … your other aggregations …
    )
)

print("Max reviews seen:", shops_r["review_count"].max())  # should be ≤ 5 now


In [ ]:
# Find places with >5 reviews
shops_r[shops_r["review_count"] > 5][["place_id", "name", "review_count"]]


In [ ]:
# Look at the actual review texts for one
pid = shops_r.loc[shops_r["review_count"] > 5, "place_id"].iloc[0]
for idx, review in enumerate(shops_r.loc[shops_r["place_id"] == pid, "text"].values[0], start=1):
    print(f"{idx}. {review}\n")


### Text Cleaning and Preprocessing

In [ ]:
import re

def clean_review_text(text):
    """Normalize review text for NLP."""
    text = str(text).lower()                          # lowercase
    text = re.sub(r"https?://\S+|www\.\S+", " ", text) # remove URLs
    text = re.sub(r"@\w+", " ", text)                  # remove mentions
    text = re.sub(r"#\w+", " ", text)                  # remove hashtags
    text = re.sub(r"[^\w\s]", " ", text)               # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()           # normalize spaces
    return text

# Create a new column with cleaned reviews for each shop
shops_r["clean_texts"] = shops_r["text"].apply(lambda reviews: [clean_review_text(r) for r in reviews])
shops_r.head()


In [ ]:
shops_r.head()

### Tokenisation

In [ ]:
import re

# A basic stopword list (can be expanded later)
STOPWORDS = set("""
a about above after again against all am an and any are as at be because been before being below
between both but by can did do does doing down during each few for from further had has have having
he her here hers herself him himself his how i if in into is it its itself just me more most my
myself no nor not of off on once only or other our ours ourselves out over own same she should so
some such than that the their theirs them themselves then there these they this those through to too
under until up very was we were what when where which while who whom why will with you your yours
yourself yourselves
""".split())

def tokenize_and_remove_stopwords(text):
    """Split text into tokens and remove common stopwords."""
    tokens = re.findall(r"[a-z']+", text)  # words only
    tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens


Clean empty reviews

In [ ]:
# Remove shops where any review text in the list is empty or NaN
shops_r["clean_texts"] = shops_r["clean_texts"].apply(
    lambda reviews: [r for r in reviews if r.strip() not in ("", "nan")]
)

# Drop rows where the resulting list is empty (no valid reviews left)
shops_r = shops_r[shops_r["clean_texts"].apply(len) > 0].copy()


In [ ]:
# Apply to each review in every shop
shops_r["tokens"] = shops_r["clean_texts"].apply(
    lambda reviews: [tokenize_and_remove_stopwords(r) for r in reviews]
)

shops_r.head(1)


In [ ]:
# Look at tokens for a random shop
import random
sample_tokens = random.choice(shops_r["tokens"].values)
for i, review_tokens in enumerate(sample_tokens, start=1):
    print(f"Review {i}: {review_tokens}")


Drop single character tokens

In [ ]:
import re

def clean_tokens(tokens):
    out = []
    for t in tokens:
        if t == "s":               # drop possessive leftovers
            continue
        if len(t) < 2:             # drop 1-char tokens
            continue
        if re.fullmatch(r"\d+", t):# drop pure numbers
            continue
        out.append(t)
    return out

# apply to each review’s token list
shops_r["tokens_clean"] = shops_r["tokens"].apply(lambda reviews: [clean_tokens(toks) for toks in reviews])
shops_r = shops_r.drop('tokens', axis=1)
shops_r.head(1)


**Lemmatization (Normalize)**

In [ ]:
# Requires NLTK once; run these two lines only the first time
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(t) for t in tokens]

shops_r["tokens_lemma"] = shops_r["tokens_clean"].apply(
    lambda reviews: [lemmatize_tokens(toks) for toks in reviews]
)
shops_r = shops_r.drop('tokens_clean', axis=1)
shops_r.head(1)


## Sentiment analysis

In [ ]:
!pip install vaderSentiment


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# initialize analyzer
analyzer = SentimentIntensityAnalyzer()


In [ ]:
def score_review(text):
    """Return compound score from VADER (-1 to +1)."""
    return analyzer.polarity_scores(text)["compound"]

# Apply to each shop: one list of review scores per place
shops_r["sentiment_scores"] = shops_r["clean_texts"].apply(
    lambda reviews: [score_review(r) for r in reviews]
)

shops_r.head(1)


In [ ]:
import numpy as np

def aggregate_sentiment(scores):
    """Return average, min, max, and percentage of positive reviews."""
    avg = np.mean(scores) if scores else 0
    pos_pct = sum(s >= 0.05 for s in scores) / len(scores) * 100 if scores else 0
    neg_pct = sum(s <= -0.05 for s in scores) / len(scores) * 100 if scores else 0
    return avg, pos_pct, neg_pct

shops_r[["avg_sentiment", "pct_positive", "pct_negative"]] = shops_r["sentiment_scores"].apply(
    lambda scores: pd.Series(aggregate_sentiment(scores))
)

shops_r.head(3)


In [ ]:
for idx, review in enumerate(shops_r.loc[shops_r["place_id"] == "ChIJ-00UHBxKDW0RT8IBRwA5KsM", "tokens_lemma"].values[0], start=1):
    print(f"{idx}. {review}\n")



In [ ]:
# Define a function to assign sentiment label based on percentages
def label_sentiment(row):
    if row["pct_positive"] > row["pct_negative"]:
        return "positive"
    elif row["pct_negative"] > row["pct_positive"]:
        return "negative"
    else:
        return "neutral"

# Apply function to create new column
shops_r["sentiment_label"] = shops_r.apply(label_sentiment, axis=1)

# Preview updated DataFrame
shops_r[["avg_sentiment", "pct_positive", "pct_negative", "sentiment_label"]].head(3)


In [ ]:
# Ensure clean_text column is a string (join tokens if it's a list)
shops_r["clean_texts"] = shops_r["clean_texts"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else str(x)
)

In [ ]:
# --- 1) Define lexicons: food stoplist and aspect terms (expand as needed) ---
food_terms = {
    # Common foods/drinks
    "pizza","burger","burgers","fries","chips","sandwich","wrap","sushi","ramen","noodles",
    "pasta","steak","chicken","beef","pork","lamb","fish","salmon","tuna","prawn","shrimp",
    "dumpling","dumplings","bao","biryani","curry","paneer","naan","taco","tacos","burrito",
    "kebab","falafel","shawarma","sausage","toast","bread","bagel","muffin","cake","pastry",
    "donut","doughnut","cookie","biscuit","pancake","waffle","icecream","ice-cream","gelato",
    "dessert","soup","salad","rice","noodle","pho","udon","soba","bibimbap","kimchi",
    "coffee","latte","cappuccino","espresso","americano","mocha","tea","chai","matcha",
    "beer","wine","cocktail","mocktail","drink","drinks","beverage","beverages","juice","smoothie",
    # Menu-ish words
    "menu","dish","dishes","meal","meals","plate","plates","portion","portions","sauce","sauces",
    "spice","spicy","sweet","sour","salty","umami"
}

# Aspect adjectives (seed) – expand with your domain words
aspect_adj_seed = {
    "great","excellent","amazing","fantastic","lovely","friendly","helpful","professional","welcoming",
    "clean","spotless","tidy","hygienic","dirty","filthy","messy",
    "fast","quick","prompt","efficient","slow","sluggish",
    "noisy","quiet","peaceful","calm","loud","crowded","busy","packed","spacious","cramped",
    "affordable","cheap","reasonable","expensive","pricey","overpriced",
    "reliable","unreliable","rude","polite","attentive","unattentive","approachable",
    "cozy","cozy","comfortable","uncomfortable","beautiful","gorgeous","nice","pleasant","poor",
    "disappointing","terrible","awful","bad","good","decent","average","outstanding","superb",
    "safe","unsafe","secure","sketchy","dodgy","vibrant","boring","dull"
}

# Optional aspect nouns that still convey “aspects” (kept to a minimum)
aspect_nouns = {
    "service","staff","ambience","atmosphere","cleanliness","vibe","value","pricing","queue","wait",
    "parking","location","access","noise","crowd","decor","seating","toilets","bathroom","hygiene"
}

# Heuristic: adjective-like suffixes (helps catch words like "friendly", "helpful", "pleasant")
adj_suffixes = ("y","ive","ous","able","ible","ful","less","al","ic","ary","ant","ent","ate","ing","ed")

# --- 2) Build an aspect-only vocabulary from the whole corpus ---
import re

def tokenize(s):
    return re.findall(r"[a-zA-Z][a-zA-Z\-']{1,}", s.lower())

def is_aspect_token(tok):
    if tok in food_terms:
        return False
    if tok in aspect_adj_seed or tok in aspect_nouns:
        return True
    # adjective-ish heuristic (avoid keeping obvious food terms accidentally)
    if len(tok) >= 3 and tok.endswith(adj_suffixes) and tok not in food_terms:
        return True
    return False

# Collect candidate aspect tokens across the corpus
candidate_vocab = set()
for text in shops_r["clean_texts"].fillna(""):
    for t in tokenize(text):
        if is_aspect_token(t):
            candidate_vocab.add(t)

# If you also want simple bigrams like "very clean", "super friendly"
def aspect_bigrams(tokens):
    bigs = []
    intensifiers = {"very","super","really","quite","extremely"}
    for a, b in zip(tokens, tokens[1:]):
        if a in intensifiers and is_aspect_token(b):
            bigs.append(f"{a} {b}")
    return bigs

# Add bigrams
for text in shops_r["clean_texts"].fillna(""):
    toks = tokenize(text)
    for bg in aspect_bigrams(toks):
        candidate_vocab.add(bg)

# Guard in case the set is empty
if not candidate_vocab:
    candidate_vocab = aspect_adj_seed | aspect_nouns

# --- 3) TF-IDF restricted to aspect vocabulary only ---
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(
    vocabulary=list(candidate_vocab),  # restrict to aspect terms only
    stop_words=None,
    ngram_range=(1, 2),
    min_df=1,          # we’ve already restricted vocab; keep 1
    max_features=None
)

X = vectorizer.fit_transform(shops_r["clean_texts"].fillna(""))
feature_names = np.array(vectorizer.get_feature_names_out())

def top_k_terms_for_row(row_vector, k=6):
    if row_vector.nnz == 0:
        return []
    nz_idx = row_vector.indices
    nz_data = row_vector.data
    top_local = np.argsort(nz_data)[-k:][::-1]
    top_feat_idx = nz_idx[top_local]
    return feature_names[top_feat_idx].tolist()

shops_r["top_keywords"] = [
    ", ".join(top_k_terms_for_row(X[i], k=6)) for i in range(X.shape[0])
]

# Inspect
shops_r[["place_id","sentiment_label","top_keywords"]].head(10)


In [ ]:
# Save DataFrame to Excel
output_file = "shops_sentiment.xlsx"
shops_r.to_excel(output_file, index=False)

print(f"✅ DataFrame saved to {output_file}")

**TUNE VADER WITH DOMAIN BASED LEXICON**

In [ ]:
# A2) Extend VADER lexicon with domain-specific weights (negatives stronger)
domain_updates = {
    "expensive": -2.2,
    "overpriced": -2.6,
    "pricey": -1.8,
    "small": -0.7,          # smaller portions in food context
    "portion": -0.5,
    "portions": -0.7,
    "funky": -2.4,          # off-taste / smell
    "stale": -2.6,
    "complaint": -2.2,
    "complaints": -2.4,
    "inedible": -3.2,
    "cold": -1.6,           # cold food (contextual; adjust if false positives)
    "bland": -2.0,
    "greasy": -1.6,
}
analyzer.lexicon.update(domain_updates)


In [ ]:
# A3) Helper to score one review with tuned VADER
def vader_score(text: str) -> float:
    return analyzer.polarity_scores(str(text))["compound"]

# quick test on your review
sample_text = """['small', 'portion', 'expensive', 'get', 'also', 'got', 'stir', 'fried', 'chicken', 'dish', 'tasted', 'funky', 'like', 'seafood',
        'bite', 'didn', 'want', 'keep', 'eating', 'tried', 'giving', 'feedback', 'phone', 'said', 'never', 'complaint']"""
vader_score(sample_text)
